# 学生英语朗读自动评测（CrisperWhisper 逐字转写版 · Google Colab）

本 notebook 用 **CrisperWhisper 2.0** 的逐字（verbatim）转写替换原来的 OpenAI Whisper。原版 Whisper 会把重复、口吃、"um/uh"这些不流利成分"顺"掉，CrisperWhisper 会照实写下来，例如：

`[um] he needs to to wear a bl- black uniform`

评测从两方面给出反馈：

- **准确度**：错读、漏读、多读。重复和自我修正**不再算作"多读"**，不扣准确度。
- **流利度**：语速、停顿、长停顿位置，以及**重复、自我修正、断词、填充词**的次数和位置。

**使用步骤**

1. 菜单 **代码执行程序 / Runtime → 更改运行时类型 / Change runtime type → T4 GPU**。CPU 也能跑，只是慢很多。
2. 在「1. 参数配置」里选择数据来源：直接从 GitHub 仓库拉取（默认）、Google Drive，或手动上传。
3. **代码执行程序 / Runtime → 全部运行 / Run all**。报告输出到 `output/` 目录，最后会自动打包下载。

**数据约定**（与原仓库一致）：朗读原文放在 `original.txt`，学生录音放在 `audio/` 目录，**文件名即学生姓名**，支持 mp3 / m4a / wav / flac / ogg / aac / mp4 / webm。

> 许可提示：CrisperWhisper 的推理代码是 MIT 许可，但**模型权重是非商业研究许可**（Nyra Health Non-Commercial Research License）。教学和科研使用没问题；商业用途需要联系 Nyra 获取授权。

## 0. 安装依赖

In [ ]:
# Colab 已自带 torch / transformers / ffmpeg，这里只需安装 CrisperWhisper（transformers 后端）
!pip install -q "crisperwhisper[transformers]>=2.1.0"
!ffmpeg -version | head -n 1

## 1. 参数配置

In [ ]:
# ===================== 数据来源 =====================
# "github" ：从 GitHub 仓库克隆（仓库里要有 original.txt 和 audio/）
# "drive"  ：使用 Google Drive 中的文件夹（报告也会写回 Drive，永久保存）
# "upload" ：运行时手动上传 original.txt 和录音文件
DATA_SOURCE = "github"
GITHUB_REPO = "https://github.com/djlei/speecheval"
DRIVE_DIR   = "/content/drive/MyDrive/speecheval"   # DATA_SOURCE="drive" 时使用

# ===================== 模型 =====================
# "turbo"（最快，推荐）/ "medium"（质量与速度折中）/ "large"（质量最好，最慢）/ "small"
MODEL_SIZE = "turbo"
USE_CACHE  = True     # 已转写过的录音直接读取缓存（*_words.json），只重新评分；想重新转写就改成 False

# ===================== 评分参数 =====================
PAUSE_THRESHOLD      = 0.3    # 相邻两个词间隔超过该值（秒）记为一次停顿
LONG_PAUSE_THRESHOLD = 1.0    # 超过该值（秒）记为长停顿，并在报告中标出位置
LOW_CONF_THRESHOLD   = 0.5    # 识别置信度低于该值的词，列为"可能发音不清晰"（仅供参考）
ACCURACY_WEIGHT      = 0.9    # 综合得分中准确度的权重（与原版一致）
FLUENCY_WEIGHT       = 0.1    # 综合得分中流利度的权重

## 2. 准备数据

In [ ]:
import os, subprocess, shutil

AUDIO_EXTS = (".mp3", ".m4a", ".wav", ".flac", ".ogg", ".aac", ".mp4", ".webm")

if DATA_SOURCE == "github":
    BASE_DIR = "/content/speecheval"
    if not os.path.isdir(BASE_DIR):
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, BASE_DIR], check=True)
elif DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = DRIVE_DIR
elif DATA_SOURCE == "upload":
    from google.colab import files
    BASE_DIR = "/content/speecheval_upload"
    os.makedirs(os.path.join(BASE_DIR, "audio"), exist_ok=True)
    print("请选择 original.txt 和所有学生录音（可多选）：")
    for name in files.upload():
        if name.lower().endswith(".txt"):
            shutil.move(name, os.path.join(BASE_DIR, "original.txt"))
        elif name.lower().endswith(AUDIO_EXTS):
            shutil.move(name, os.path.join(BASE_DIR, "audio", name))
else:
    raise ValueError(f"未知的 DATA_SOURCE：{DATA_SOURCE}")

ORIGINAL_TXT = os.path.join(BASE_DIR, "original.txt")
AUDIO_DIR    = os.path.join(BASE_DIR, "audio")
OUTPUT_DIR   = os.path.join(BASE_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.isfile(ORIGINAL_TXT), f"找不到原文文件：{ORIGINAL_TXT}"
audio_files = sorted(f for f in os.listdir(AUDIO_DIR) if f.lower().endswith(AUDIO_EXTS))
assert audio_files, f"{AUDIO_DIR} 下没有找到录音文件"
print(f"✅ 数据目录：{BASE_DIR}")
print(f"✅ 共 {len(audio_files)} 个录音：{', '.join(audio_files)}")

## 3. 工具函数

In [ ]:
import re, json, csv, string, difflib
from collections import Counter
import numpy as np

PUNCT = string.punctuation + "’‘“”…—–"
_PUNCT_TABLE = str.maketrans("", "", PUNCT)
FILLERS = {"um", "umm", "uh", "uhm", "er", "erm", "ah", "eh", "hmm", "mm", "mhm"}


def normalize_word(w):
    # 小写、去标点（含撇号，o'clock → oclock，原文和识别结果处理方式一致）
    return w.lower().translate(_PUNCT_TABLE).strip()


def preprocess_text(text):
    # 原文标准化：连字符视为空格，再逐词标准化
    text = re.sub(r"[-–—]", " ", text)
    return [n for n in (normalize_word(w) for w in text.split()) if n]


def fmt_time(sec):
    if sec is None:
        return "--:--"
    return f"{int(sec // 60):02d}:{sec % 60:04.1f}"


# ---------------------------------------------------------------
# 音频解码：用 ffmpeg 转成 16 kHz 单声道（soundfile 不支持 m4a/mp3 等格式）
# ---------------------------------------------------------------
def load_audio_16k(path):
    cmd = ["ffmpeg", "-nostdin", "-v", "error", "-i", path,
           "-f", "f32le", "-ac", "1", "-ar", "16000", "-"]
    out = subprocess.run(cmd, capture_output=True, check=True).stdout
    return np.frombuffer(out, np.float32).copy()


# ---------------------------------------------------------------
# 把 CrisperWhisper 输出的每个词归类：
#   word   普通单词        filler  填充词 [um] [uh]
#   cutoff 断词 "th-"      sound   其他声音 [laughter] [cough] ...
# ---------------------------------------------------------------
def classify_tokens(raw_words, ref_vocab):
    tokens = []
    for w in raw_words:
        raw = (w.get("word") or "").strip()
        start, end, prob = w.get("start"), w.get("end"), w.get("prob")
        t = raw.strip(" .,!?;:\"“”'‘’").lower()
        if not t:
            continue
        if t.startswith("[") and t.endswith("]"):
            inner = t[1:-1].strip()
            kind = "filler" if inner in FILLERS else "sound"
            tokens.append(dict(kind=kind, text=f"[{inner}]", norm=inner, start=start, end=end, prob=prob))
            continue
        if len(t) > 1 and t.endswith("-"):
            tokens.append(dict(kind="cutoff", text=t, norm=normalize_word(t), start=start, end=end, prob=prob))
            continue
        # 普通词：内部连字符拆开（如 well-known），时间平均分配
        parts = [p for p in (normalize_word(x) for x in re.split(r"[-\s]+", t)) if p]
        if not parts:
            continue
        span = (end - start) / len(parts) if (start is not None and end is not None) else None
        for k, p in enumerate(parts):
            s = start + k * span if span is not None else start
            e = s + span if span is not None else end
            kind = "filler" if (p in FILLERS and p not in ref_vocab) else "word"
            tokens.append(dict(kind=kind, text=p, norm=p, start=s, end=e, prob=prob))
    return tokens


# ---------------------------------------------------------------
# 词级最小编辑距离对齐（比 difflib 更适合做朗读比对）
# 返回 [(tag, ref_idx, hyp_idx)]，tag ∈ equal / replace / insert / delete
# insert 的 ref_idx 表示"插在原文第 ref_idx 个词之前"
# ---------------------------------------------------------------
def align_words(ref, hyp):
    n, m = len(ref), len(hyp)
    D = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        D[i][0] = i
    for j in range(1, m + 1):
        D[0][j] = j
    for i in range(1, n + 1):
        ri = ref[i - 1]
        for j in range(1, m + 1):
            c = 0 if ri == hyp[j - 1] else 1
            D[i][j] = min(D[i - 1][j - 1] + c, D[i - 1][j] + 1, D[i][j - 1] + 1)
    i, j, pairs = n, m, []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and D[i][j] == D[i - 1][j - 1] + (0 if ref[i - 1] == hyp[j - 1] else 1):
            pairs.append(("equal" if ref[i - 1] == hyp[j - 1] else "replace", i - 1, j - 1))
            i, j = i - 1, j - 1
        elif j > 0 and D[i][j] == D[i][j - 1] + 1:
            pairs.append(("insert", i, j - 1))
            j -= 1
        else:
            pairs.append(("delete", i - 1, None))
            i -= 1
    return pairs[::-1]


def group_segments(pairs):
    # 把连续的同类操作合并成段：dict(tag, ref=[...], hyp=[...], pos)
    segs = []
    for tag, ri, hj in pairs:
        last = segs[-1] if segs else None
        if last and last["tag"] == tag and tag != "equal" and (tag != "insert" or last["pos"] == ri):
            if ri is not None and tag != "insert":
                last["ref"].append(ri)
            if hj is not None:
                last["hyp"].append(hj)
        else:
            segs.append(dict(tag=tag, pos=ri,
                             ref=[] if (tag == "insert" or ri is None) else [ri],
                             hyp=[] if hj is None else [hj]))
    return segs


def _contains(seq, sub):
    k = len(sub)
    return any(seq[i:i + k] == sub for i in range(len(seq) - k + 1))


def classify_insert(seg, ref, hyp):
    # 判断一段"多出来的词"是重复、自我修正还是真正的多读
    S = [hyp[j] for j in seg["hyp"]]
    k, p = len(S), seg["pos"]
    j1, j2 = seg["hyp"][0], seg["hyp"][-1] + 1
    # 1) 与紧邻的实际读出内容相同 → 重复（如 to to、he needs to he needs to）
    if hyp[max(0, j1 - k):j1] == S or hyp[j2:j2 + k] == S:
        return "repeat"
    # 2) 是附近原文的一段 → 回读/重复
    if _contains(ref[max(0, p - k - 2):p + k + 2], S):
        return "repeat"
    # 3) 单个词且与相邻原文词很像（如 need → needs）→ 自我修正
    if k == 1:
        for r in ref[max(0, p - 1):p + 1]:
            if (len(S[0]) >= 2 and (r.startswith(S[0]) or S[0].startswith(r))) \
                    or difflib.SequenceMatcher(None, S[0], r).ratio() >= 0.7:
                return "selfcorr"
    return "extra"

In [ ]:
# ---------------------------------------------------------------
# 核心评测：对齐 + 准确度 + 流利度指标 + 标注文本
# ---------------------------------------------------------------
def evaluate(raw_words, ref_words):
    ref_vocab = set(ref_words)
    tokens = classify_tokens(raw_words, ref_vocab)
    core_idx = [i for i, t in enumerate(tokens) if t["kind"] == "word"]   # 参与对齐的词
    hyp = [tokens[i]["norm"] for i in core_idx]

    pairs = align_words(ref_words, hyp)
    segs = group_segments(pairs)

    labels = {}            # 对齐用的 hyp 下标 → equal/replace/repeat/selfcorr/extra
    errors, repeats, selfcorrs = [], [], []
    last_t = None          # 上一个已读词的结束时间，用来给漏读标注大致位置
    for s in segs:
        t0 = tokens[core_idx[s["hyp"][0]]]["start"] if s["hyp"] else last_t
        if s["hyp"]:
            last_t = tokens[core_idx[s["hyp"][-1]]]["end"]
        if s["tag"] == "equal":
            for j in s["hyp"]:
                labels[j] = "equal"
        elif s["tag"] == "replace":
            for j in s["hyp"]:
                labels[j] = "replace"
            errors.append(("错读", t0, f"原文「{' '.join(ref_words[i] for i in s['ref'])}」"
                                        f" → 读成「{' '.join(hyp[j] for j in s['hyp'])}」"))
        elif s["tag"] == "delete":
            errors.append(("漏读", t0, f"原文「{' '.join(ref_words[i] for i in s['ref'])}」未朗读"))
        else:  # insert
            kind = classify_insert(s, ref_words, hyp)
            for j in s["hyp"]:
                labels[j] = kind
            text = " ".join(hyp[j] for j in s["hyp"])
            if kind == "repeat":
                repeats.append((t0, text))
            elif kind == "selfcorr":
                selfcorrs.append((t0, text))
            else:
                errors.append(("多读", t0, f"额外读出「{text}」"))

    correct = sum(1 for tg, _, _ in pairs if tg == "equal")
    accuracy = correct / len(ref_words) * 100 if ref_words else 0.0

    # ---------- 标注文本（按实际朗读顺序，漏读插在对应位置） ----------
    core_pos = {ti: j for j, ti in enumerate(core_idx)}
    timed_prev_end = None
    long_pauses = []
    out, emitted = [], 0

    def emit_token(ti):
        nonlocal timed_prev_end
        t = tokens[ti]
        if t["start"] is not None and timed_prev_end is not None:
            gap = t["start"] - timed_prev_end
            if gap > LONG_PAUSE_THRESHOLD:
                out.append(f"‖{gap:.1f}s‖")
                prev = next((tokens[x]["text"] for x in range(ti - 1, -1, -1)), "")
                long_pauses.append((timed_prev_end, gap, prev, t["text"]))
        if t["end"] is not None:
            timed_prev_end = t["end"]
        if t["kind"] == "word":
            lab = labels.get(core_pos[ti], "equal")
            out.append({"equal": t["text"], "replace": f"【{t['text']}】",
                        "repeat": f"⟨重复:{t['text']}⟩", "selfcorr": f"⟨修正:{t['text']}⟩",
                        "extra": f"{{多读:{t['text']}}}"}[lab])
        elif t["kind"] == "cutoff":
            out.append(f"⟨断词:{t['text']}⟩")
        else:
            out.append(t["text"])

    for tg, ri, hj in pairs:
        if hj is None:                       # 漏读
            out.append(f"（漏:{ref_words[ri]}）")
            continue
        ti = core_idx[hj]
        while emitted < ti:                  # 先输出夹在中间的填充词/断词/声音
            if tokens[emitted]["kind"] != "word":
                emit_token(emitted)
            emitted += 1
        if tg == "replace":
            # 错读：同时显示原词
            t = tokens[ti]
            emit_token(ti)
            out[-1] = f"【{ref_words[ri]}→{t['text']}】"
        else:
            emit_token(ti)
        emitted = ti + 1
    while emitted < len(tokens):
        if tokens[emitted]["kind"] != "word":
            emit_token(emitted)
        emitted += 1
    annotated = " ".join(out)
    # 合并连续的重复标注：⟨重复:who⟩ ⟨重复:help⟩ → ⟨重复:who help⟩
    while True:
        merged = re.sub(r"⟨重复:([^⟩]+)⟩ ⟨重复:([^⟩]+)⟩", r"⟨重复:\1 \2⟩", annotated)
        if merged == annotated:
            break
        annotated = merged

    # ---------- 流利度指标 ----------
    timed = sorted((t for t in tokens if t["start"] is not None and t["end"] is not None),
                   key=lambda t: t["start"])
    if timed:
        total_duration = timed[-1]["end"] - timed[0]["start"]
        speech_time = sum(t["end"] - t["start"] for t in timed if t["kind"] in ("word", "cutoff"))
        gaps = [timed[i]["start"] - timed[i - 1]["end"] for i in range(1, len(timed))]
    else:
        total_duration, speech_time, gaps = 0.0, 0.0, []
    pauses = [g for g in gaps if g > PAUSE_THRESHOLD]
    n_fillers = sum(1 for t in tokens if t["kind"] == "filler")
    n_cutoffs = sum(1 for t in tokens if t["kind"] == "cutoff")
    effective_words = sum(1 for lab in labels.values() if lab in ("equal", "replace"))
    n_disfl = len(repeats) + len(selfcorrs) + n_cutoffs + n_fillers
    minutes = total_duration / 60 if total_duration > 0 else 0

    fluency_data = {
        "总朗读时长(秒)": round(total_duration, 2),
        "有效发音时长(秒)": round(speech_time, 2),
        "停顿次数(>%.1fs)" % PAUSE_THRESHOLD: len(pauses),
        "平均停顿时长(秒)": round(sum(pauses) / len(pauses), 2) if pauses else 0,
        "长停顿(>%.1fs)次数" % LONG_PAUSE_THRESHOLD: len(long_pauses),
        "语速(WPM，不含重复/填充词)": round(effective_words / minutes, 1) if minutes else 0,
        "每分钟正确词数(WCPM)": round(correct / minutes, 1) if minutes else 0,
        "语音占比(%)": round(speech_time / total_duration * 100, 1) if total_duration else 0,
        "重复次数": len(repeats),
        "自我修正次数": len(selfcorrs),
        "断词次数": n_cutoffs,
        "填充词次数(um/uh等)": n_fillers,
        "每百词不流利次数": round(n_disfl / len(ref_words) * 100, 1) if ref_words else 0,
    }

    low_conf = [(t["start"], t["text"], t["prob"]) for j, ti in enumerate(core_idx)
                if (t := tokens[ti])["prob"] is not None and t["prob"] < LOW_CONF_THRESHOLD
                and labels.get(j) == "equal"]

    return dict(tokens=tokens, accuracy=accuracy, correct=correct, errors=errors,
                repeats=repeats, selfcorrs=selfcorrs, long_pauses=long_pauses,
                annotated=annotated, fluency_data=fluency_data, low_conf=low_conf)

In [ ]:
def fluency_grade(fd, repeats):
    # 流利度评分（满分 100）：语速 25 + 语音占比 20 + 长停顿 25 + 不流利 30
    wpm = fd["语速(WPM，不含重复/填充词)"]
    ratio = fd["语音占比(%)"]
    long_pause = fd["长停顿(>%.1fs)次数" % LONG_PAUSE_THRESHOLD]
    disfl = fd["每百词不流利次数"]
    fb, score = [], 0

    if wpm >= 100:   fb.append("✅ 语速较快，朗读节奏流畅"); score += 25
    elif wpm >= 70:  fb.append("✅ 语速适中，节奏平稳自然"); score += 22
    elif wpm >= 50:  fb.append("⚠️ 语速偏慢，存在较多思考间隙"); score += 15
    else:            fb.append("❌ 语速过慢，连贯性不足"); score += 8

    if ratio >= 70:   fb.append("✅ 有效发音占比高，停顿很少"); score += 20
    elif ratio >= 55: fb.append("✅ 发音占比正常，停顿合理"); score += 17
    elif ratio >= 40: fb.append("⚠️ 停顿偏多，流畅度有待提升"); score += 12
    else:             fb.append("❌ 停顿过多，朗读断断续续"); score += 6

    if long_pause == 0:   fb.append("✅ 无明显长停顿，整体连贯"); score += 25
    elif long_pause <= 2: fb.append(f"⚠️ 存在{long_pause}处长停顿，对流畅度影响不大"); score += 18
    elif long_pause <= 5: fb.append(f"❌ 存在{long_pause}处长停顿，流畅度受明显影响"); score += 10
    else:                 fb.append(f"❌ 长停顿多达{long_pause}处，流畅度较差"); score += 5

    detail = (f"重复{fd['重复次数']}次、自我修正{fd['自我修正次数']}次、"
              f"断词{fd['断词次数']}次、填充词{fd['填充词次数(um/uh等)']}次")
    if disfl <= 2:    fb.append(f"✅ 几乎没有重复和卡顿（{detail}）"); score += 30
    elif disfl <= 5:  fb.append(f"⚠️ 有少量重复或卡顿（{detail}）"); score += 22
    elif disfl <= 10: fb.append(f"❌ 重复和卡顿较多（{detail}）"); score += 14
    else:             fb.append(f"❌ 重复和卡顿频繁（{detail}），建议先熟读课文"); score += 6

    common = Counter(text for _, text in repeats).most_common(3)
    if common and common[0][1] >= 2:
        fb.append("💡 反复重复的地方：" + "、".join(f"「{w}」×{c}" for w, c in common if c >= 2))

    level = "优秀" if score >= 85 else "良好" if score >= 70 else "一般" if score >= 50 else "待加强"
    return {"流利度得分": score, "流利度等级": level, "详细评价": fb}


def total_comment(total):
    if total >= 90: return "整体表现优秀，发音准确且朗读流畅，继续保持。"
    if total >= 75: return "整体表现良好，针对少量错误和停顿加强练习即可明显提升。"
    if total >= 60: return "整体表现一般，建议先纠正易错单词，再逐步提升朗读流畅度。"
    return "整体有待加强，建议从单词发音基础开始练习，熟练后再提升语速。"


def generate_report(name, verbatim_text, ev, ref_words, fl, total):
    L = []
    L += ["=" * 60, "学生朗读评估报告", f"学生姓名：{name}", f"转写模型：CrisperWhisper 2.0 ({MODEL_SIZE}, verbatim)", "=" * 60, ""]
    L += ["【一、逐字转写（保留重复、填充词、断词）】", verbatim_text.strip(), ""]
    L += ["【二、朗读标注】",
          "  图例：【原词→读成】错读  （漏:词）漏读  {多读:词}  ⟨重复:词⟩  ⟨修正:词⟩  ⟨断词:th-⟩  [um]填充词  ‖1.5s‖长停顿",
          ev["annotated"], ""]
    L += ["【三、发音准确度评估】",
          f"原文总单词数：{len(ref_words)}",
          f"正确朗读单词数：{ev['correct']}",
          f"准确率：{ev['accuracy']:.1f}%（重复和自我修正不扣准确度分）", "", "错误明细："]
    if ev["errors"]:
        for kind, t, msg in ev["errors"]:
            icon = {"错读": "❌", "漏读": "⚠️", "多读": "➕"}[kind]
            L.append(f"  {icon} {kind}{'' if t is None else f' [{fmt_time(t)}]'}：{msg}")
    else:
        L.append("  ✅ 全部匹配，未发现错读、漏读、多读")
    L.append("")
    L.append("【四、流利度指标详情】")
    L += [f"  {k}：{v}" for k, v in ev["fluency_data"].items()]
    if ev["repeats"] or ev["selfcorrs"]:
        L.append("  重复 / 自我修正位置：")
        items = [(t, "重复", w) for t, w in ev["repeats"]] + [(t, "修正", w) for t, w in ev["selfcorrs"]]
        for t, kind, w in sorted(items, key=lambda x: x[0] if x[0] is not None else 0):
            L.append(f"    [{fmt_time(t)}] {kind}「{w}」")
    if ev["long_pauses"]:
        L.append("  长停顿位置：")
        for t, gap, prev, nxt in ev["long_pauses"]:
            L.append(f"    [{fmt_time(t)}] 「{prev}」与「{nxt}」之间停顿 {gap:.1f} 秒")
    L.append("")
    L += ["【五、流利度评价】",
          f"  流利度得分：{fl['流利度得分']}/100", f"  流利度等级：{fl['流利度等级']}", "  详细说明："]
    L += [f"    {x}" for x in fl["详细评价"]]
    L.append("")
    if ev["low_conf"]:
        L.append("【六、识别置信度较低的词（可能发音不够清晰，仅供参考，建议回听确认）】")
        L.append("  " + "、".join(f"{w}[{fmt_time(t)}]" for t, w, _ in ev["low_conf"][:15]))
        L.append("")
    L += ["【七、综合总评】",
          f"  综合得分：{total:.1f}/100（准确度占{ACCURACY_WEIGHT:.0%}，流利度占{FLUENCY_WEIGHT:.0%}）",
          f"  {total_comment(total)}", "", "=" * 60]
    return "\n".join(L) + "\n"

## 4. 加载模型

In [ ]:
import torch
from crisperwhisper import CrisperWhisperModel

HAS_GPU = torch.cuda.is_available()
print("🖥️ 运行设备：", torch.cuda.get_device_name(0) if HAS_GPU else "CPU（会比较慢，建议切换到 GPU 运行时）")
print(f"🔄 正在加载 CrisperWhisper 2.0 ({MODEL_SIZE})，首次运行需要下载模型...")
model = CrisperWhisperModel(MODEL_SIZE, backend="transformers",
                            compute_type="float16" if HAS_GPU else "float32")
print("✅ 模型加载完成")

with open(ORIGINAL_TXT, "r", encoding="utf-8") as f:
    ref_words = preprocess_text(f.read())
print(f"✅ 原文读取完成，共 {len(ref_words)} 个单词")

## 5. 批量转写与评测

In [ ]:
import time

def transcribe_verbatim(audio_path, cache_path):
    if USE_CACHE and os.path.isfile(cache_path):
        with open(cache_path, encoding="utf-8") as f:
            return json.load(f), True
    audio = load_audio_16k(audio_path)
    res = model.transcribe(audio, sr=16000, language="en", mode="verbatim", word_confidence=True)
    data = {"model": MODEL_SIZE, "text": res.text,
            "words": [{"word": w.word, "start": w.start, "end": w.end, "prob": w.probability}
                      for w in (res.words or [])]}
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=1)
    return data, False


summary = []
for idx, audio_file in enumerate(audio_files, 1):
    name = os.path.splitext(audio_file)[0]
    print(f"[{idx}/{len(audio_files)}] 正在处理：{name}")
    try:
        t0 = time.time()
        data, cached = transcribe_verbatim(os.path.join(AUDIO_DIR, audio_file),
                                           os.path.join(OUTPUT_DIR, f"{name}_words.json"))
        print(f"  {'📦 读取缓存' if cached else f'🎙️ 转写完成（{time.time() - t0:.0f} 秒）'}")

        ev = evaluate(data["words"], ref_words)
        fl = fluency_grade(ev["fluency_data"], ev["repeats"])
        total = ev["accuracy"] * ACCURACY_WEIGHT + fl["流利度得分"] * FLUENCY_WEIGHT

        report = generate_report(name, data["text"], ev, ref_words, fl, total)
        with open(os.path.join(OUTPUT_DIR, f"{name}.txt"), "w", encoding="utf-8") as f:
            f.write(report)

        fd = ev["fluency_data"]
        summary.append({"学生": name, "准确率(%)": round(ev["accuracy"], 1),
                        "错误数": len(ev["errors"]), "流利度得分": fl["流利度得分"],
                        "语速WPM": fd["语速(WPM，不含重复/填充词)"], "WCPM": fd["每分钟正确词数(WCPM)"],
                        "重复": fd["重复次数"], "自我修正": fd["自我修正次数"], "断词": fd["断词次数"],
                        "填充词": fd["填充词次数(um/uh等)"],
                        "长停顿": fd["长停顿(>%.1fs)次数" % LONG_PAUSE_THRESHOLD],
                        "综合得分": round(total, 1)})
        print(f"  ✅ 准确率 {ev['accuracy']:.1f}%｜流利度 {fl['流利度得分']}｜重复 {fd['重复次数']} 次"
              f"｜综合 {total:.1f} → {name}.txt")
    except Exception as e:
        print(f"  ❌ 处理失败：{e}")

if summary:
    with open(os.path.join(OUTPUT_DIR, "summary.csv"), "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(summary[0]))
        w.writeheader()
        w.writerows(summary)
print(f"\n🎉 全部完成！报告保存在：{OUTPUT_DIR}")

## 6. 查看结果与下载

In [ ]:
import pandas as pd
from IPython.display import display

if summary:
    display(pd.DataFrame(summary))
    first = summary[0]["学生"]
    print(f"\n—— 示例报告：{first} ——\n")
    print(open(os.path.join(OUTPUT_DIR, f"{first}.txt"), encoding="utf-8").read())

In [ ]:
# 打包下载 output 目录（使用 Google Drive 时报告已经保存在 Drive 中，可跳过）
zip_path = shutil.make_archive("/content/speecheval_reports", "zip", OUTPUT_DIR)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("不在 Colab 环境中，压缩包位于：", zip_path)